# Chronos-2 Quantisation

In [1]:
import os
from pathlib import Path

import torch

os.environ.setdefault("HOME", os.environ.get("USERPROFILE", str(Path.home())))

from chop.models import get_model
from chop.ir.graph import MaseGraph
from chop.passes.graph import PASSES


c:\Users\johnn\Documents\coding\mase\.venv\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
c:\Users\johnn\Documents\coding\mase\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\johnn\Documents\coding\mase\.venv\Lib\site-packages\timm\models\helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


In [2]:
# Basic run config
OUTPUT_DIR = Path("artifacts")
GRAPH_NAME = "chronos2_quantized"
DEVICE = "cpu"
WRITE_QUANT_SUMMARY = False  # Set True to generate comparison CSVs (slower).

# Conservative starting quantisation config (only core arithmetic ops)
QUANT_PASS_ARGS = {
    "by": "type",
    "default": {"config": {"name": None}},
    "linear": {
        "config": {
            "name": "integer",
            "data_in_width": 8,
            "data_in_frac_width": 4,
            "weight_width": 8,
            "weight_frac_width": 4,
            "bias_width": 8,
            "bias_frac_width": 4,
        }
    },
    "matmul": {
        "config": {
            "name": "integer",
            "data_in_width": 8,
            "data_in_frac_width": 4,
            "weight_width": 8,
            "weight_frac_width": 4,
        }
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# 1) Load model
model = get_model("chronos-2", pretrained=True)
model.eval()
if DEVICE:
    if DEVICE.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but torch.cuda.is_available() is False.")
    model = model.to(DEVICE)

# Force eager attention so FX sees matmul/add/softmax nodes.
if hasattr(model.config, "_attn_implementation"):
    model.config._attn_implementation = "eager"

print("Loaded Chronos-2 model")

Loaded Chronos-2 model


In [4]:
# 2) Build graph + required metadata
batch_size = 1

print(model.config.chronos_config)
c_len = model.config.chronos_config["context_length"]
out_patch = model.config.chronos_config["output_patch_size"]

dummy_in = {
    "context": torch.randn((batch_size, c_len)),
    "group_ids": torch.zeros((batch_size,), dtype=torch.long),
    "future_covariates": torch.zeros((batch_size, out_patch), dtype=torch.float32),
    "num_output_patches": 1,
}

mg = MaseGraph(
    model,
    hf_input_names=[
        "context",
        "group_ids",
        "future_covariates",
        "num_output_patches",
    ],
)

mg, _ = PASSES["init_metadata"](mg)
mg, _ = PASSES["add_common_metadata"](mg, pass_args={"dummy_in": dummy_in})

print(f"Graph nodes before quantisation: {sum(1 for _ in mg.nodes)}")

`past_key_values` were not specified as input names, but model.config.use_cache = True. Setting model.config.use_cache = False.


{'context_length': 8192, 'input_patch_size': 16, 'input_patch_stride': 16, 'max_output_patches': 64, 'output_patch_size': 16, 'quantiles': [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99], 'time_encoding_scale': 8192, 'use_arcsinh': True, 'use_reg_token': True}
tensor([[-1.2188,  0.0557,  1.1128,  ..., -0.3318, -0.0638, -0.3328]])
tensor([[-1.2188,  0.0557,  1.1128,  ..., -0.3318, -0.0638, -0.3328]])
tensor([[False, False, False,  ..., False, False, False]])
tensor([[True, True, True,  ..., True, True, True]])
tensor([[-1.2188,  0.0557,  1.1128,  ..., -0.3318, -0.0638, -0.3328]])
tensor([[-1.2188,  0.0557,  1.1128,  ..., -0.3318, -0.0638, -0.3328]])
tensor([[-1.2123,  0.0622,  1.1193,  ..., -0.3253, -0.0573, -0.3263]])
tensor([[1.4697, 0.0039, 1.2528,  ..., 0.1058, 0.0033, 0.1065]])
tensor([[0.9964]])
tensor([[-1.0252,  0.0623,  0.9646,  ..., -0.3204, -0.0574, -0.3214]])
tensor([[-1.0252,  0.0623,  0.9646,  ..., -0.32

In [5]:
# 3) Quantise
orig_mg = None
if WRITE_QUANT_SUMMARY:
    # Needed only for before/after comparison in summarize_quantization.
    orig_mg = MaseGraph(model, hf_input_names=["context", "group_ids", "future_covariates", "num_output_patches"])
    orig_mg, _ = PASSES["init_metadata"](orig_mg)
    orig_mg, _ = PASSES["add_common_metadata"](orig_mg, pass_args={"dummy_in": dummy_in})

mg, _ = PASSES["quantize"](mg, pass_args=QUANT_PASS_ARGS)
print("Quantisation pass complete")

if WRITE_QUANT_SUMMARY:
    PASSES["summarize_quantization"](
        mg,
        {"save_dir": OUTPUT_DIR / "quantize_summary", "original_graph": orig_mg},
    )
    print("Saved quantization summary to artifacts/quantize_summary")


Quantisation pass complete


In [6]:
# 4) Export quantised graph
base = OUTPUT_DIR / GRAPH_NAME
mg.export(str(base))
print(f"Exported: {base}.pt and {base}.mz")

INFO     Exporting MaseGraph to artifacts\chronos2_quantized.pt, artifacts\chronos2_quantized.mz
INFO     Exporting GraphModule to artifacts\chronos2_quantized.pt
INFO     Saving full model format
INFO     Exporting MaseMetadata to artifacts\chronos2_quantized.mz
WARNING  Failed to pickle call_function node: finfo
WARNING  cannot pickle 'torch.finfo' object
WARNING  Failed to pickle call_function node: getattr_8
WARNING  cannot pickle 'torch.finfo' object
WARNING  Failed to pickle call_function node: finfo_1
WARNING  cannot pickle 'torch.finfo' object
WARNING  Failed to pickle call_function node: getattr_10
WARNING  cannot pickle 'torch.finfo' object


Exported: artifacts\chronos2_quantized.pt and artifacts\chronos2_quantized.mz


In [7]:
# Perform bench 
import fev
from chronos import Chronos2Pipeline

# Add chronos_config to mg.model
setattr(mg.model, "config", model.config)
setattr(mg.model, "chronos_config", model.chronos_config)
setattr(mg.model, "device", model.device)

# The GraphModule returns a plain dict; wrap forward() to restore Chronos2Output type
from chop.models.chronos2.modeling_chronos2 import Chronos2Output

_orig_forward = mg.model.forward
def _patched_forward(*args, **kwargs):
    output = _orig_forward(*args, **kwargs)
    if isinstance(output, dict) and not isinstance(output, Chronos2Output):
        return Chronos2Output(**output)
    return output
mg.model.forward = _patched_forward

# Define the benchmark task
tasks_configs = [
    {
        "dataset_path": "autogluon/chronos_datasets",
        "dataset_config": "monash_m1_quarterly",
        "horizon": 8,
        "seasonality": 4,
        "eval_metric": "MASE",
    },
    {
        "dataset_path": "autogluon/chronos_datasets",
        "dataset_config": "monash_electricity_weekly",
        "horizon": 8,
        "num_windows": 2,
    },
    {
        "dataset_path": "autogluon/chronos_datasets",
        "dataset_config": "monash_m1_yearly",
        "horizon": 6,
    },
]
benchmark = fev.Benchmark.from_list(tasks_configs)

summaries = []

In [8]:
# Run evaluation with mg.model
import tqdm
pipeline = Chronos2Pipeline(model=mg.model)

for task in tqdm.tqdm(benchmark.tasks, desc="Evaluating tasks"):
    predictions_per_window, inference_time_s = pipeline.predict_fev(task, batch_size=256)
    print(f"Inference time: {inference_time_s:.2f}s")
    summary = task.evaluation_summary(predictions_per_window, "quantised_chronos2")
    summaries.append(summary)


Evaluating tasks:  33%|███▎      | 1/3 [00:20<00:40, 20.38s/it]

Inference time: 3.38s


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Evaluating tasks:  67%|██████▋   | 2/3 [00:45<00:23, 23.36s/it]

Inference time: 12.31s


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Evaluating tasks: 100%|██████████| 3/3 [01:00<00:00, 20.09s/it]

Inference time: 1.88s


In [9]:
# Predict with pre-quantisation model for comparison
pipeline = Chronos2Pipeline(model=model)
for task in tqdm.tqdm(benchmark.tasks, desc="Evaluating tasks with pre-quantisation model"):
    predictions_per_window, inference_time_s = pipeline.predict_fev(task, batch_size=256)
    print(f"Inference time (pre-quantisation model): {inference_time_s:.2f}s")
    summary = task.evaluation_summary(predictions_per_window, "pre_quantisation_chronos2")
    summaries.append(summary)

Evaluating tasks with pre-quantisation model:  33%|███▎      | 1/3 [00:03<00:06,  3.35s/it]

Inference time (pre-quantisation model): 3.23s


Evaluating tasks with pre-quantisation model:  67%|██████▋   | 2/3 [00:13<00:07,  7.51s/it]

Inference time (pre-quantisation model): 10.14s


Evaluating tasks with pre-quantisation model: 100%|██████████| 3/3 [00:16<00:00,  5.42s/it]

Inference time (pre-quantisation model): 2.39s


In [10]:
fev.leaderboard(summaries=summaries, baseline_model="pre_quantisation_chronos2")

,win_rate,skill_score,median_training_time_s,median_inference_time_s,training_corpus_overlap,num_failures
model_name,,,,,,
pre_quantisation_chronos2,1.0,0.000000,NaN,NaN,0.0,0
quantised_chronos2,0.0,-0.324376,NaN,NaN,0.0,0


In [12]:
# Print test errors for each task
for summary in summaries:
    print(f"Model: {summary['model_name']} Dataset: {summary['dataset_config']} test_error: {summary['test_error']:.4f}")

Model: quantised_chronos2 Dataset: monash_m1_quarterly test_error: 4.5088
Model: quantised_chronos2 Dataset: monash_electricity_weekly test_error: 2.5492
Model: quantised_chronos2 Dataset: monash_m1_yearly test_error: 9.7965
Model: pre_quantisation_chronos2 Dataset: monash_m1_quarterly test_error: 2.9556
Model: pre_quantisation_chronos2 Dataset: monash_electricity_weekly test_error: 2.2295
Model: pre_quantisation_chronos2 Dataset: monash_m1_yearly test_error: 7.3562
